In [1]:
# Cài các thư viện cần dùng. Colab cần Java 21 cho Pyserini.
import sys, subprocess

if "google.colab" in sys.modules:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-21-jdk-headless"], check=True)

# Cài đặt Pillow 10.4.0 trước để đảm bảo tính tương thích trên Python 3.13
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "Pillow==10.4.0"
], check=True)

# Cài đặt pyserini==0.22.1 để tương thích với định dạng index cast2019 (Lucene 8.x)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "pyserini==0.22.1", "ir-datasets==0.6.3", "ir-measures==0.4.3",
    "pandas", "tqdm", "ollama"
], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'pyserini==0.22.1', 'ir-datasets==0.6.3', 'ir-measures==0.4.3', 'pandas', 'tqdm', 'ollama'], returncode=0)

In [24]:
# Tự động cài đặt faiss-cpu nếu chưa có để sửa lỗi ModuleNotFoundError: No module named 'faiss'
import sys, subprocess
try:
    import faiss
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"], check=True)

# Cấu hình: Ollama Cloud trên Colab, không dùng localhost:11434.
# Đã đồng bộ cấu hình tương thích cho Pillow trên Python 3.13.
from pathlib import Path
import json
import os

import ir_datasets
import ir_measures
import pandas as pd
from ir_measures import nDCG, Recall, RR
from ollama import Client
from pyserini.search.lucene import LuceneSearcher
from tqdm.auto import tqdm

MODEL = "gemma4:31b"  # tên model cloud; đổi nếu muốn, xem https://ollama.com/search?c=cloud
OLLAMA_HOST = "https://ollama.com"
TOP_K = 100
K1, B = 0.9, 0.4

OUTPUT = Path("cast2020_results")
OUTPUT.mkdir(exist_ok=True)

api_key = "4c9948c61df9434b86ff83570ef9bf4e.XOwZkjfbF1XNg4mEeAZfSBLR"

client = Client(
    host=OLLAMA_HOST,
    headers={"Authorization": f"Bearer {api_key}"},
)
del api_key

print({"model": MODEL, "ollama_host": OLLAMA_HOST, "top_k": TOP_K})


{'model': 'gemma4:31b', 'ollama_host': 'https://ollama.com', 'top_k': 100}


In [2]:
# Đọc 216 lượt để tạo history; chỉ 208 lượt có qrels được dùng để chấm điểm.
full = ir_datasets.load("trec-cast/v1/2020")
judged_dataset = ir_datasets.load("trec-cast/v1/2020/judged")

qrels = pd.DataFrame([
    {"query_id": q.query_id, "doc_id": q.doc_id, "relevance": q.relevance}
    for q in judged_dataset.qrels_iter()
])
judged_ids = set(qrels.query_id)

history_by_topic = {}
turns = []
for q in sorted(full.queries_iter(), key=lambda x: (x.topic_number, x.turn_number)):
    history = history_by_topic.setdefault(q.topic_number, [])
    turns.append({
        "query_id": q.query_id,
        "topic": q.topic_number,
        "turn": q.turn_number,
        "raw": q.raw_utterance,
        "human": q.manual_rewritten_utterance,
        "history": history.copy(),
    })
    history.append(q.raw_utterance)

turns = pd.DataFrame(turns)
judged = turns[turns.query_id.isin(judged_ids)].reset_index(drop=True)

assert len(turns) == 216
assert len(judged) == 208
assert set(judged.query_id) == judged_ids

print(f"{len(turns)} total turns, {len(judged)} judged turns, {len(qrels):,} qrels")
judged.head(3)

[INFO] [starting] https://trec.nist.gov/data/cast/2020qrels.txt
[INFO] [finished] https://trec.nist.gov/data/cast/2020qrels.txt: [00:00] [1.56MB] [3.53MB/s]
[INFO] [starting] https://raw.githubusercontent.com/daltonj/treccastweb/master/2020/2020_manual_evaluation_topics_v1.0.json
[INFO] [finished] https://raw.githubusercontent.com/daltonj/treccastweb/master/2020/2020_manual_evaluation_topics_v1.0.json: [00:00] [79.0kB] [20.9MB/s]
                                                                                                                                      

216 total turns, 208 judged turns, 40,451 qrels


,query_id,topic,turn,raw,human,history
0,81_1,81,1,How do you know when your garage door opener i...,How do you know when your garage door opener i...,[]
1,81_2,81,2,Now it stopped working. Why?,Now my garage door opener stopped working. Why?,[How do you know when your garage door opener ...
2,81_3,81,3,How much does it cost for someone to fix it?,How much does it cost for someone to repair a ...,[How do you know when your garage door opener ...


In [3]:
# Pyserini tự tải, kiểm tra và giải nén prebuilt index cast2019.
searcher = LuceneSearcher.from_prebuilt_index("cast2019", verbose=True)
searcher.set_bm25(K1, B)

assert searcher.num_docs == 38_429_835

# Cổng an toàn: mọi document ID trong qrels phải tồn tại trong index.
missing = [
    doc_id
    for doc_id in tqdm(qrels.doc_id.unique(), desc="Checking qrel document IDs")
    if searcher.doc(doc_id) is None
]
assert not missing, f"Qrels và index không tương thích; ví dụ: {missing[:5]}"

smoke_hits = searcher.search(judged.iloc[0].raw, k=3)
assert smoke_hits
[(hit.docid, hit.score) for hit in smoke_hits]

Attempting to initialize pre-built index cast2019.


index-cast2019.tar.gz: 19.8GB [21:18, 16.6MB/s]                            


Extracting /root/.cache/pyserini/indexes/index-cast2019.tar.gz into /root/.cache/pyserini/indexes/index-cast2019.36e604d7f5a4e08ade54e446be2f6345...
{'total_terms': 1593628213, 'documents': 38429835, 'non_empty_documents': 38426205, 'unique_terms': -1}
Index passes consistency checks against pre-built index 'cast2019'!
Initializing cast2019...


Checking qrel document IDs:   0%|          | 0/29264 [00:00<?, ?it/s]

[('MARCO_6154874', 23.265274047851562),
 ('MARCO_6568085', 21.818201065063477),
 ('MARCO_8454278', 21.524202346801758)]

In [4]:
# Hai hàm nhỏ dùng chung cho cả Raw, LLM và Human.
MEASURES = [nDCG @ 10, Recall @ 100, RR(rel=1) @ 10]

def retrieve(queries, name):
    rows = []
    for query_id, text in tqdm(queries.items(), desc=f"Retrieving {name}"):
        for rank, hit in enumerate(searcher.search(text, k=TOP_K), start=1):
            rows.append({
                "query_id": query_id,
                "doc_id": hit.docid,
                "rank": rank,
                "score": hit.score,
                "method": name,
            })
    return pd.DataFrame(rows)

def evaluate(run):
    aggregate = {
        str(measure): score
        for measure, score in ir_measures.calc_aggregate(MEASURES, qrels, run).items()
    }
    per_query = pd.DataFrame([
        {"query_id": result.query_id, "metric": str(result.measure), "value": result.value}
        for result in ir_measures.iter_calc(MEASURES, qrels, run)
    ]).pivot(index="query_id", columns="metric", values="value")
    return aggregate, per_query.reindex(judged.query_id).fillna(0)


In [5]:
# Baselines: Raw và Human chạy trước, không liên quan tới Ollama.
query_table = judged.set_index("query_id")
runs, scores, per_query = {}, {}, {}

for method, column in {"raw": "raw", "human": "human"}.items():
    runs[method] = retrieve(query_table[column], method)
    scores[method], per_query[method] = evaluate(runs[method])
    runs[method].to_csv(OUTPUT / f"{method}_run.csv", index=False)

pd.DataFrame(scores).T

Retrieving raw: 0it [00:00, ?it/s]

Retrieving human: 0it [00:00, ?it/s]

,R@100,nDCG@10,RR@10
raw,0.125089,0.077960,0.172014
human,0.427310,0.255781,0.542960


In [25]:
# Định nghĩa các SYSTEM PROMPT cho từng phiên bản rewriter
SYSTEM_PROMPTS = {
    "rewriter 1": """Rewrite the current conversational search utterance as one standalone search query.
Use only facts from the preceding user utterances. Preserve intent, entities, numbers,
dates, constraints, negation, and language. Do not answer the question or explain.
Return only the rewritten query on one line.""",

    "rewriter 2": """You rewrite conversational questions into search queries for BM25 retrieval,
then append a small set of contextual synonyms to improve lexical matching.

INPUT
- history: preceding user questions, ordered from oldest to newest.
- current_query: the user's current question.

Use current_query and history as the evidence for resolving conversational
context. History contains questions, not answers. A previous question does
not establish that its proposed answer or assumption is true.

You may use linguistic knowledge to generate equivalent expressions for
concepts already present in the rewritten query. This does not authorize
adding new contextual facts or guessing missing referents.

WORKFLOW

1. Identify the current information need.
   Identify the subject, the aspect being asked about, and explicit
   constraints such as time, location, comparison targets, and exclusions.
   Treat current_query as the primary source of the user's intent.

2. Check whether context is missing.
   Identify references or omitted details that prevent the current question
   from being understood independently.
   If the question is already self-contained, use it unchanged as the
   base query and proceed to keyword expansion.

3. Resolve missing context selectively.
   For each missing detail, find supporting wording in history.
   Prefer the most recent context that fits the current question.
   Use earlier context when the question clearly returns to it.
   Carry forward only the subject and constraints that still apply.
   When the current question changes an earlier constraint, use the current
   constraint.

   If history does not identify a unique referent, retain the supported
   wording or use a supported descriptive phrase. Leave unresolved details
   unspecified rather than selecting an unsupported entity.

4. Construct and finalize the base query.
   Preserve the current question's informative words wherever possible.
   Replace resolved references with their explicit subjects and insert
   necessary omitted context.
   Preserve the current aspect, entities, numbers, dates, constraints,
   negation, and language.
   Make only the wording changes needed for a readable standalone query.

   Verify that the base query asks for the same information as current_query,
   that added contextual details are supported, and that earlier questions
   have not become additional search objectives.
   Remove any answer, guessed fact, or speculative subtopic.

   Once finalized, keep this base query unchanged during the remaining steps.

5. Generate contextual synonym candidates.
   Identify alternative expressions for concepts in the base query.
   Consider synonyms, unambiguous alternative names, and full forms of
   abbreviations whose meanings are clear from the base query.
   Prefer expressions that a relevant passage could use to express the
   same concept.

6. Filter and select expansion keywords.
   Select zero to three terms or short phrases that:
   - Express the same concept in this specific context.
   - Add an alternative expression absent from the base query.
   - Preserve the information need, scope, polarity, and constraints.

   Exclude possible answers, guessed facts, new entities, speculative
   causes, related subtopics, and broader or narrower concepts.
   Exclude duplicates and trivial singular, plural, or tense variations.
   Leave ambiguous references unresolved.
   Use the same language as the base query, except for established names
   or abbreviations.

   Prefer precision over filling the available slots.
   If no reliable expansion exists, select no keywords.

7. Assemble the final search query.
   Copy the finalized base query exactly.
   Append the selected keywords after it, separated by spaces.
   If no keywords were selected, return the base query alone.

OUTPUT
Return only the final search query on one line.
Do not include labels, JSON, Markdown, explanations, or answers.
"""
}

CACHE = OUTPUT / "ollama_rewrites.csv"
# Không handle hay đọc từ cache cũ - reset mới hoàn toàn mỗi lần chạy
results_list = []

# Chạy từng lượt viết lại cho mỗi cấu hình prompt
total_steps = len(judged) * len(SYSTEM_PROMPTS)
with tqdm(total=total_steps, desc="Ollama Cloud rewriting") as pbar:
    for method_name, prompt_text in SYSTEM_PROMPTS.items():
        for turn in judged.itertuples(index=False):
            user_input = json.dumps({
                "history": turn.history,
                "current_query": turn.raw,
            }, ensure_ascii=False)

            response = client.chat(
                model=MODEL,
                messages=[
                    {"role": "system", "content": prompt_text},
                    {"role": "user", "content": user_input},
                ],
                think=False,
                options={"temperature": 0, "num_predict": 128},
            )

            rewrite = (response.message.content or "").strip()
            if not rewrite or "\n" in rewrite:
                print(f"Warning: Output không hợp lệ tại {turn.query_id} ({method_name}): {rewrite!r}")
                rewrite = turn.raw

            results_list.append({
                "query_id": turn.query_id,
                "rewrite": rewrite,
                "model": MODEL,
                "method": method_name
            })
            pbar.update(1)

rewrites = pd.DataFrame(results_list)
rewrites.to_csv(CACHE, index=False)
display(rewrites.head())


Ollama Cloud rewriting:   0%|          | 0/416 [00:00<?, ?it/s]

,query_id,rewrite,model,method
0,81_1,signs of a failing garage door opener,gemma4:31b,rewriter 1
1,81_2,Why would a garage door opener stop working?,gemma4:31b,rewriter 1
2,81_3,How much does it cost to have a broken garage ...,gemma4:31b,rewriter 1
3,81_4,How much does it cost to replace a garage door...,gemma4:31b,rewriter 1
4,81_5,How do I choose a new garage door opener?,gemma4:31b,rewriter 1


In [35]:
# Thực hiện Retrieval và Evaluation cho toàn bộ các phương pháp thành công
# Dọn sạch hoàn toàn các biến để loại bỏ mọi phiên bản rewriter cũ còn lưu trong bộ nhớ
runs = {}
scores = {}
per_query = {}

# Tính toán lại baseline gốc cố định từ đầu
for method, column in {"raw": "raw", "human": "human"}.items():
    runs[method] = retrieve(query_table[column], method)
    scores[method], per_query[method] = evaluate(runs[method])
    runs[method].to_csv(OUTPUT / f"{method}_run.csv", index=False)

# Chạy đánh giá cho các rewriter mới có trong SYSTEM_PROMPTS hiện tại
for method_name in SYSTEM_PROMPTS.keys():
    method_rewrites = rewrites[rewrites["method"] == method_name]
    if method_rewrites.empty:
        continue

    llm_queries = method_rewrites.set_index("query_id").rewrite.reindex(judged.query_id)
    runs[method_name] = retrieve(llm_queries, method_name)
    scores[method_name], per_query[method_name] = evaluate(runs[method_name])
    runs[method_name].to_csv(OUTPUT / f"{method_name}_run.csv", index=False)

# Tạo bảng kết quả tổng hợp chính xác bao gồm các phương pháp đang kích hoạt
active_methods = ["raw"] + [m for m in SYSTEM_PROMPTS.keys() if m in scores] + ["human"]
summary = pd.DataFrame(scores).T.loc[active_methods]
summary.to_csv(OUTPUT / "aggregate_metrics.csv")
display(summary)

# Tiến hành xây dựng bảng so sánh chi tiết cho cả rewriter 1 và rewriter 2 so với raw
if "rewriter 1" in per_query and "rewriter 2" in per_query:
    # Lấy các bản rewrite
    rewrites_r1 = rewrites[rewrites["method"] == "rewriter 1"][["query_id", "rewrite"]].rename(columns={"rewrite": "rewriter_1_rewrite"})
    rewrites_r2 = rewrites[rewrites["method"] == "rewriter 2"][["query_id", "rewrite"]].rename(columns={"rewrite": "rewriter_2_rewrite"})

    # Tính delta cho rewriter 1
    delta_r1 = per_query["rewriter 1"] - per_query["raw"]
    delta_r1["r1_classification"] = delta_r1["nDCG@10"].map(
        lambda x: "improved" if x > 0 else "degraded" if x < 0 else "tied"
    )
    delta_r1 = delta_r1.rename(columns={
        "R@100": "r1_delta_R@100",
        "RR@10": "r1_delta_RR@10",
        "nDCG@10": "r1_delta_nDCG@10"
    })

    # Tính delta cho rewriter 2
    delta_r2 = per_query["rewriter 2"] - per_query["raw"]
    delta_r2["r2_classification"] = delta_r2["nDCG@10"].map(
        lambda x: "improved" if x > 0 else "degraded" if x < 0 else "tied"
    )
    delta_r2 = delta_r2.rename(columns={
        "R@100": "r2_delta_R@100",
        "RR@10": "r2_delta_RR@10",
        "nDCG@10": "r2_delta_nDCG@10"
    })

    # Ghép bảng tổng hợp chi tiết
    delta_detailed = (judged[["query_id", "topic", "turn", "raw", "human"]]
                      .merge(rewrites_r1, on="query_id")
                      .merge(rewrites_r2, on="query_id")
                      .merge(delta_r1.reset_index(), on="query_id")
                      .merge(delta_r2.reset_index(), on="query_id"))

    delta_detailed.to_csv(OUTPUT / "llm_minus_raw_per_query.csv", index=False)
    print("Đã lưu bảng so sánh chi tiết của cả 2 rewriters vào file llm_minus_raw_per_query.csv")
    display(delta_detailed.head(5))

Retrieving raw: 0it [00:00, ?it/s]

Retrieving human: 0it [00:00, ?it/s]

Retrieving rewriter 1: 0it [00:00, ?it/s]

Retrieving rewriter 2: 0it [00:00, ?it/s]

,R@100,nDCG@10,RR@10
raw,0.125089,0.077960,0.172014
rewriter 1,0.369924,0.229223,0.510464
rewriter 2,0.387319,0.241809,0.512363
human,0.427310,0.255781,0.542960


Đã lưu bảng so sánh chi tiết của cả 2 rewriters vào file llm_minus_raw_per_query.csv


,query_id,topic,turn,raw,human,rewriter_1_rewrite,rewriter_2_rewrite,r1_delta_R@100,r1_delta_RR@10,r1_delta_nDCG@10,r1_classification,r2_delta_R@100,r2_delta_RR@10,r2_delta_nDCG@10,r2_classification
0,81_1,81,1,How do you know when your garage door opener i...,How do you know when your garage door opener i...,signs of a failing garage door opener,How do you know when your garage door opener i...,-0.044444,-0.5,-0.165227,degraded,0.044444,-0.5,-0.102679,degraded
1,81_2,81,2,Now it stopped working. Why?,Now my garage door opener stopped working. Why?,Why would a garage door opener stop working?,Why did the garage door opener stop working? g...,0.530612,1.0,0.360258,improved,0.469388,0.5,0.352922,improved
2,81_3,81,3,How much does it cost for someone to fix it?,How much does it cost for someone to repair a ...,How much does it cost to have a broken garage ...,How much does it cost for someone to fix a gar...,0.533333,1.0,0.279704,improved,0.933333,0.5,0.286209,improved
3,81_4,81,4,How about replacing it instead?,How much does it cost to replace a garage door...,How much does it cost to replace a garage door...,cost to replace garage door opener replacement...,0.473684,1.0,0.196572,improved,0.578947,0.5,0.231924,improved
4,81_5,81,5,How do I choose a new one?,How do I choose a new garage door opener?,How do I choose a new garage door opener?,How do I choose a new garage door opener? sele...,0.694444,0.5,0.329115,improved,0.638889,1.0,0.267433,improved


In [37]:
import shutil

# Nén lại thư mục kết quả mới nhất
output_filename = "cast2020_results"
shutil.make_archive(output_filename, 'zip', OUTPUT)

print(f"Đã cập nhật và tạo thành công file zip mới: {output_filename}.zip")

Đã cập nhật và tạo thành công file zip mới: cast2020_results.zip
